In [1]:
import json
import urllib.request
from pathlib import Path
from typing import Dict, List, Any


from sentence_transformers import SentenceTransformer

MODEL_NAME = "all-MiniLM-L6-v2"
BATCH_SIZE = 256

NODES_URL = "https://raw.githubusercontent.com/ryosuzuki/lecture-demo/refs/heads/main/sample-data/nodes.json"

NODES_PATH = Path("nodes.json")
OUT_PATH = Path("embeddings.json")

def download_nodes(url: str, out_path: Path) -> None:
    # Simple, dependency-free download
    with urllib.request.urlopen(url) as r:
        data = r.read()
    out_path.write_bytes(data)

def load_nodes() -> List[Dict[str, Any]]:
    data = json.loads(NODES_PATH.read_text(encoding="utf-8"))
    return list(data.values()) if isinstance(data, dict) else data

def extract_embedding_keys(nodes: List[Dict[str, Any]]) -> List[str]:
    keys: List[str] = []
    for n in nodes:
        k = n.get("embedding_key") or n.get("description") or n.get("text")
        if isinstance(k, str) and k.strip():
            keys.append(k.strip())
    seen = set()
    uniq = []
    for k in keys:
        if k not in seen:
            seen.add(k)
            uniq.append(k)
    return uniq

def main():
    # download
    download_nodes(NODES_URL, NODES_PATH)
    print("Downloaded:", NODES_PATH.resolve(), "bytes:", NODES_PATH.stat().st_size)

    nodes = load_nodes()
    keys = extract_embedding_keys(nodes)
    print(f"Loaded nodes: {len(nodes)} | unique embedding keys: {len(keys)}")

    model = SentenceTransformer(MODEL_NAME)
    embeddings = {}

    for i in range(0, len(keys), BATCH_SIZE):
        batch = keys[i:i+BATCH_SIZE]
        vecs = model.encode(
            batch,
            batch_size=64,
            show_progress_bar=False,
            normalize_embeddings=False
        )
        for k, v in zip(batch, vecs):
            embeddings[k] = v.tolist()
        print(f"Embedded {min(i+BATCH_SIZE, len(keys))}/{len(keys)}")

    OUT_PATH.write_text(json.dumps(embeddings, indent=2), encoding="utf-8")
    print("Done. Wrote:", OUT_PATH.resolve())

    # files.download("embeddings.json")

main()

c:\Users\srina\projects\Generative Agents and Memory-Based Retrieval\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Downloaded: C:\Users\srina\projects\Generative Agents and Memory-Based Retrieval\nodes.json bytes: 309103
Loaded nodes: 593 | unique embedding keys: 202


Loading weights: 100%|██████████| 103/103 [00:00<00:00, 776.21it/s, Materializing param=pooler.dense.weight]                             
BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Embedded 202/202
Done. Wrote: C:\Users\srina\projects\Generative Agents and Memory-Based Retrieval\embeddings.json


In [2]:
import json
from pathlib import Path

data = json.loads(Path("embeddings.json").read_text(encoding="utf-8"))
print("Number of items:", len(data))

first_key = next(iter(data))
print("Vector dimension:", len(data[first_key]))

print("First 20 keys:")
for i, k in enumerate(list(data.keys())[:20], 1):
    print(i, k)

Number of items: 202
Vector dimension: 384
First 20 keys:
1 bed is being used
2 Klaus Mueller is sleeping
3 This is Klaus Mueller's plan for Tuesday February 14: wake up and complete the morning routine at 7:00 am, go to the library at Oak Hill College and work on his research paper from 8:00 am to 1:00 pm, have lunch at Hobbs Cafe at 12:00 pm, continue work at the library from 2:00 pm to 5:00 pm, Eat dinner at 5:00 pm, Watch TV for an hour from 7:00 pm to 8:00 pm, read a Sociology-related book from 8:00 pm to 9:00 pm.
4 bed is being slept on
5 Klaus Mueller is getting ready for bed
6 bed is idle
7 game console is idle
8 closet is idle
9 desk is idle
10 cooking area is idle
11 refrigerator is idle
12 kitchen sink is idle
13 toaster is idle
14 reflecting on the book
15 library table is in use
16 library sofa is idle
17 library table is idle
18 bookshelf is idle
19 common room sofa is idle
20 pool table is idle
